# ML Methods: Platform Use vs. Daily Mood

**Goal:** Apply simple machine learning models to see whether daily platform usage (minutes) can *predict* daily well-being scores.

**Important note about ML here:** The dataset has only **21 days**, so results are **very sensitive** and mainly exploratory.
We use **cross-validation** to avoid over-interpreting a single train/test split.

## 1) Imports
We use pandas/numpy for data handling, and scikit-learn for modeling and evaluation.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import make_scorer, mean_absolute_error

## 2) Load the dataset
I load the same CSV used in the EDA and hypothesis testing steps.

In [ ]:
import os

# Try common filenames so the notebook works even if you rename the CSV in your repo.
possible_files = [
    "daily_social_media.csv",
    "daily_social_media (1).csv",
    "daily_social_media.csv".replace(" ", ""),
]

csv_file = None
for f in possible_files:
    if os.path.exists(f):
        csv_file = f
        break

if csv_file is None:
    raise FileNotFoundError(
        "Could not find the dataset CSV. Put it in the same folder as this notebook and name it "
        "'daily_social_media.csv' (recommended)."
    )

df = pd.read_csv(csv_file)
df["date"] = pd.to_datetime(df["date"])

print("Loaded:", csv_file)
df.head()

**Result (quick check):** If the table above looks correct (dates + minutes + scores), the data loaded successfully.

## 3) Define features (X) and targets (y)
**X (inputs):** minutes spent on each platform  
**y (outputs):** anxiety, social comparison, productivity, motivation

In [ ]:
feature_cols = ["ig_min", "tw_min", "tt_min", "yt_min"]
target_cols = ["anxiety", "soc_comp", "productivity", "motivation"]

X = df[feature_cols].values
targets = {col: df[col].values for col in target_cols}

print("Rows:", len(df))
print("Features:", feature_cols)
print("Targets:", target_cols)

**Result:** This prints the number of rows (days) and confirms which columns are used for ML.

## 4) Set up models and evaluation
We compare a few very common regression models:

- **Baseline (mean):** predicts the average score (a sanity check)
- **Linear Regression:** simplest model
- **Ridge / Lasso:** linear models with regularization (helpful with small data)
- **Random Forest:** a non-linear model (can capture interactions, but may overfit on small data)

**Metrics**
- **MAE (Mean Absolute Error):** average absolute prediction error (lower is better)
- **R²:** variance explained (higher is better; can be negative if a model is worse than baseline)

In [ ]:
# Cross-validation setup (5 folds is a common choice; small data -> keep it simple)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "MAE": make_scorer(mean_absolute_error, greater_is_better=False),  # negative inside sklearn
    "R2": "r2"
}

models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0, random_state=42))]),
    "Lasso Regression": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.01, random_state=42, max_iter=10000))]),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42)
}

**Result:** Models are defined. Next, we evaluate them consistently using the same cross-validation setting.

## 5) Evaluate models for each target
This produces a table of cross-validated MAE and R² values for each outcome variable.

In [ ]:
def evaluate_target(y):
    rows = []
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        mae = -scores["test_MAE"].mean()
        r2 = scores["test_R2"].mean()
        rows.append((name, mae, r2))
    out = pd.DataFrame(rows, columns=["Model", "CV_MAE (lower=better)", "CV_R2 (higher=better)"])
    return out.sort_values("CV_MAE (lower=better)")

all_tables = {}
for tname, y in targets.items():
    all_tables[tname] = evaluate_target(y)
    print("\n=== Target:", tname, "===\n")
    display(all_tables[tname])

**Result (how to read):**
- The **top row** in each table is the model with the **lowest average MAE** (best by this metric).
- If **R² is near 0 or negative**, it means the model does not predict that target well from platform minutes alone (common with small/noisy data).

## 6) Fit an interpretable model and look at coefficients (Ridge)
For interpretation, I fit **Ridge Regression** with standardized features.
This lets us compare platforms on the same scale.

**Interpretation rule (standardized features):**
- Positive coefficient → more minutes tends to increase the score
- Negative coefficient → more minutes tends to decrease the score

(These are associations, not causation.)

In [ ]:
ridge_pipe = Pipeline([("scaler", StandardScaler()),
                       ("model", Ridge(alpha=1.0, random_state=42))])

coef_table = []

for tname, y in targets.items():
    ridge_pipe.fit(X, y)
    coefs = ridge_pipe.named_steps["model"].coef_
    for feat, c in zip(feature_cols, coefs):
        coef_table.append([tname, feat, c])

coef_df = pd.DataFrame(coef_table, columns=["target", "feature", "ridge_coef (standardized)"])
coef_df

**Result:** The table shows which platform minutes are most strongly associated with each outcome in a simple linear model.

## 7) Visualize coefficients for easier comparison
A quick bar chart per target.

In [ ]:
for tname in target_cols:
    sub = coef_df[coef_df["target"] == tname].set_index("feature")["ridge_coef (standardized)"]
    plt.figure()
    sub.plot(kind="bar")
    plt.title(f"Ridge coefficients (standardized) for {tname}")
    plt.ylabel("Coefficient")
    plt.xticks(rotation=0)
    plt.show()

**Result:** Bars farther from 0 indicate a stronger relationship *in the linear model*.

## 8) Short summary of what I found (from ML part)
This summary is **data-driven** and based on the cross-validation tables and ridge coefficients.

- If most models show **negative/near-zero R²**, it means platform minutes alone are **not enough** to reliably predict that outcome for this dataset.
- If one target shows **positive R²** and lower MAE, it means minutes might have some predictive signal for that target.

In [ ]:
# Create a small "best model" summary by target using MAE
best_summary = []
for tname, table in all_tables.items():
    best_row = table.iloc[0]
    best_summary.append([tname, best_row["Model"], best_row["CV_MAE (lower=better)"], best_row["CV_R2 (higher=better)"]])

best_df = pd.DataFrame(best_summary, columns=["target", "best_model_by_MAE", "best_CV_MAE", "best_CV_R2"])
best_df

**Result:** This table shows the best-performing model *by MAE* for each target.
In my dataset, the strongest predictive performance is expected to be limited because the dataset is small (21 days).

## 9) Limitations (ML-specific)
- **Very small dataset (21 days):** high variance in model results.
- **Self-reported scores:** subjective noise is unavoidable.
- **Missing confounders:** sleep, stress, exams, weather, social events, etc. can dominate mood.
- **Minutes are not content:** what you watched/read matters, not only duration.

**Future improvements**
- Collect more days (e.g., 60–90)
- Add features (sleep, study hours, deadlines, step count, etc.)
- Track content type (educational vs entertainment) for YouTube/TikTok